In [16]:
import pandas as pd
from pathlib import Path

# Load dataset
project_root = Path.cwd().parent
data_path = project_root / "data" / "GlobalWeatherRepository.csv"

df = pd.read_csv(data_path)

df["last_updated"] = pd.to_datetime(df["last_updated"])

# Select Amman temperature
city_df = df[df["location_name"] == "Amman"].copy()

city_df = city_df.sort_values("last_updated")

daily_temperature_df = (
    city_df[["last_updated", "temperature_celsius"]]
    .set_index("last_updated")
    .resample("D")
    .mean()
    .ffill()
    .reset_index()
)

daily_temperature_df.head()


,last_updated,temperature_celsius
0,2024-05-16,24.0
1,2024-05-17,27.0
2,2024-05-18,30.0
3,2024-05-19,29.0
4,2024-05-20,30.0


In [18]:
daily_temperature_df["naive_prediction"] = (
    daily_temperature_df["temperature_celsius"].shift(1)
)
baseline_df = daily_temperature_df.dropna()

baseline_df.head()

,last_updated,temperature_celsius,naive_prediction
1,2024-05-17,27.0,24.0
2,2024-05-18,30.0,27.0
3,2024-05-19,29.0,30.0
4,2024-05-20,30.0,29.0
5,2024-05-21,29.0,30.0


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Prepare train and test data
# Prepare train and test data
split_index = int(len(daily_temperature_df) * 0.8)
train_df = daily_temperature_df.iloc[:split_index].copy()
test_df = daily_temperature_df.iloc[split_index:].copy()

train_series = train_df.set_index("last_updated")["temperature_celsius"]
test_series = test_df.set_index("last_updated")["temperature_celsius"]

# Build SARIMA model
model = SARIMAX(
    train_series,
    order=(1,1,1),
    seasonal_order=(1,1,1,7)
)

# Train model
sarima_model = model.fit()

print(sarima_model.summary())
test_series = test_df.set_index("last_updated")["temperature_celsius"]

# Build SARIMA model
model = SARIMAX(
    train_series,
    order=(1,1,1),
    seasonal_order=(1,1,1,7)
)

# Train model
sarima_model = model.fit()

print(sarima_model.summary())

NameError: name 'train_df' is not defined